In [ ]:
import re
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

from scipy.signal import correlate, resample

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    precision_score, recall_score, f1_score, accuracy_score,
    confusion_matrix, balanced_accuracy_score, cohen_kappa_score,
    matthews_corrcoef, log_loss
)
from sklearn.metrics import top_k_accuracy_score
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_auc_score, average_precision_score


# ============================================================
# CONFIG — DATASET-1 ALL SUBJECTS, PAPER-COMPATIBLE MODEL
# ============================================================

# Dataset-1 export folders should be directly here:
# /home/tsultan1/IEEE_Access/Dataset-1/final_exports-sub1
# /home/tsultan1/IEEE_Access/Dataset-1/final_exports-sub2
# ...
DATA_ROOT = Path("/home/tsultan1/IEEE_Access/Dataset-1")

SUB_DIR_PATTERN = re.compile(r"final_exports-sub(\d+)$")

OUT_DIR = Path("/home/tsultan1/IEEE_Access/Dataset-1/train_dataset1_allsubjects_NeuroFusionTrans_SYNC_PAPER_MATCHED_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Paper preprocessing setting: 1000 ms window at 200 Hz = 200 samples
EEG_CH = 8
EMG_CH = 8
TARGET_FS = 200
WINDOW_TIME_MS = 1000
EXPECTED_T = int(TARGET_FS * WINDOW_TIME_MS / 1000)

# K-fold CV — keep current window-level protocol
K_FOLDS = 5

# Paper-compatible training
EPOCHS = 150
PATIENCE = 25
BATCH_SIZE = 32
LR = 5e-4
WEIGHT_DECAY = 1e-4
GRAD_CLIP = 1.0

# Paper-compatible transformer
D_MODEL = 128
NUM_HEADS = 4
NUM_LAYERS = 2
FF_DIM = 256
DROPOUT = 0.20

# Paper cross-modality attention initialization
INIT_ALPHA_EEG = 0.7
INIT_ALPHA_EMG = 0.3

# Training augmentation
USE_TRAIN_AUGMENTATION = True
AUG_NOISE_STD = 0.01
AUG_SCALE_MIN = 0.90
AUG_SCALE_MAX = 1.10
AUG_SHIFT_MAX = 5

# Frequency perturbation augmentation
USE_FREQ_PERTURBATION = True
FREQ_PERTURB_PROB = 0.30
FREQ_PERTURB_MIN = 0.98
FREQ_PERTURB_MAX = 1.02

# Synthetic sample generation through linear combinations
USE_MIXUP = True
MIXUP_ALPHA = 0.20

# Online adaptation
DO_ONLINE_ADAPT = True

# Paper-matched: use 400 new online data points if available
ONLINE_NEW_N = 400
ONLINE_CYCLES = 5
ONLINE_BS = 16
ONLINE_LR = 1e-5

# Replay buffer size
REPLAY_N = 400

# L2 regularization during online adaptation
ADAPT_L2_LAMBDA = 1e-2

# Temporal synchronization
APPLY_CROSSCORR_SYNC = True

# 50 samples at 200 Hz = 250 ms.
# Use None for full-window lag search.
MAX_SYNC_LAG_SAMPLES = 50
SAVE_SYNC_REPORT = True

# GPU
USE_AMP = True
NUM_WORKERS = 4
PIN_MEMORY = True

SEED = 42


# ============================================================
# SEED
# ============================================================

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)


# ============================================================
# CSV HELPERS
# ============================================================

def read_numeric_csv(path: Path) -> np.ndarray:
    df = pd.read_csv(path)
    df = df.apply(pd.to_numeric, errors="coerce").fillna(0.0)
    return df.to_numpy(dtype=np.float32)


def load_labels_csv(lbl_path: Path, sid: int):
    df = pd.read_csv(lbl_path)
    df.rename(columns={c: c.strip() for c in df.columns}, inplace=True)

    label_col = None
    for c in df.columns:
        if c.lower() == "label":
            label_col = c
            break

    if label_col is None:
        raise ValueError(f"{lbl_path} missing 'Label' column")

    y = df[label_col].astype(np.int64).to_numpy()

    subj_col = None
    for c in df.columns:
        if c.lower() == "subject_id":
            subj_col = c
            break

    if subj_col is None:
        sid_arr = np.full(len(y), sid, dtype=np.int64)
    else:
        sid_arr = df[subj_col].astype(np.int64).to_numpy()

    return y, sid_arr


# ============================================================
# DATA LOADING — DATASET-1
# ============================================================

def subject_id_from_folder(folder: Path) -> int:
    m = SUB_DIR_PATTERN.match(folder.name)
    if not m:
        raise ValueError(f"Folder does not match final_exports-subX pattern: {folder}")
    return int(m.group(1))


def find_subject_files(sub_dir: Path, sid: int):
    eeg_candidates = [
        sub_dir / f"eeg_sub{sid}.csv",
        sub_dir / "eeg_sub1.csv",
        sub_dir / "eeg_final.csv",
    ]

    emg_candidates = [
        sub_dir / f"emg_sub{sid}.csv",
        sub_dir / "emg_sub1.csv",
        sub_dir / "emg_final.csv",
    ]

    lbl_candidates = [
        sub_dir / f"labels_sub{sid}.csv",
        sub_dir / "labels_sub1.csv",
        sub_dir / "labels_final.csv",
    ]

    eeg_path = next((p for p in eeg_candidates if p.exists()), None)
    emg_path = next((p for p in emg_candidates if p.exists()), None)
    lbl_path = next((p for p in lbl_candidates if p.exists()), None)

    return eeg_path, emg_path, lbl_path


def load_one_subject_folder(sub_dir: Path):
    sid = subject_id_from_folder(sub_dir)
    eeg_path, emg_path, lbl_path = find_subject_files(sub_dir, sid)

    if eeg_path is None or emg_path is None or lbl_path is None:
        print(f"[SKIP] {sub_dir.name}: missing EEG/EMG/label CSV")
        print(f"  eeg_path={eeg_path}")
        print(f"  emg_path={emg_path}")
        print(f"  lbl_path={lbl_path}")
        return None

    eeg = read_numeric_csv(eeg_path)
    emg = read_numeric_csv(emg_path)
    y, sid_arr = load_labels_csv(lbl_path, sid)

    n = min(len(y), eeg.shape[0], emg.shape[0], len(sid_arr))
    if n <= 0:
        print(f"[SKIP] {sub_dir.name}: empty after length alignment")
        return None

    eeg = eeg[:n]
    emg = emg[:n]
    y = y[:n]
    sid_arr = sid_arr[:n]

    return eeg, emg, y, sid_arr


def list_subject_export_folders(data_root: Path):
    """
    Finds Dataset-1 export folders:
        /home/tsultan1/IEEE_Access/Dataset-1/final_exports-subX
    """
    if not data_root.exists():
        raise RuntimeError(f"DATA_ROOT does not exist: {data_root}")

    subdirs = []
    for p in data_root.iterdir():
        if p.is_dir() and SUB_DIR_PATTERN.match(p.name):
            subdirs.append(p)

    subdirs = sorted(subdirs, key=lambda p: subject_id_from_folder(p))
    return subdirs


def load_all_subjects(data_root: Path):
    subdirs = list_subject_export_folders(data_root)

    print("[INFO] Found Dataset-1 export folders:")
    for sd in subdirs:
        print(" ", sd)

    if not subdirs:
        raise RuntimeError(
            f"No folders found like final_exports-subX directly under {data_root}. "
            f"Run Dataset-1 preprocessing first."
        )

    eeg_all, emg_all, y_all, sid_all = [], [], [], []
    eeg_dim_ref = None
    emg_dim_ref = None

    for sd in subdirs:
        try:
            out = load_one_subject_folder(sd)

            if out is None:
                continue

            eeg, emg, y, sid_arr = out

            if eeg_dim_ref is None:
                eeg_dim_ref = eeg.shape[1]
                emg_dim_ref = emg.shape[1]
            else:
                if eeg.shape[1] != eeg_dim_ref or emg.shape[1] != emg_dim_ref:
                    warnings.warn(
                        f"Skipping {sd.name}: feature mismatch "
                        f"EEG={eeg.shape[1]} EMG={emg.shape[1]} vs "
                        f"ref EEG={eeg_dim_ref} EMG={emg_dim_ref}"
                    )
                    continue

            eeg_all.append(eeg)
            emg_all.append(emg)
            y_all.append(y)
            sid_all.append(sid_arr)

            print(f"[OK] {sd.name}: N={len(y)} | EEG={eeg.shape} | EMG={emg.shape}")

        except Exception as e:
            warnings.warn(f"[SKIP] {sd.name}: {e}")

    if not eeg_all:
        raise RuntimeError("No subject data loaded. Check Dataset-1 export files.")

    EEG = np.vstack(eeg_all).astype(np.float32)
    EMG = np.vstack(emg_all).astype(np.float32)
    y = np.concatenate(y_all).astype(np.int64)
    sid = np.concatenate(sid_all).astype(np.int64)

    return EEG, EMG, y, sid


def reshape_flat_windows(EEG, EMG):
    if EEG.shape[1] % EEG_CH != 0:
        raise ValueError(
            f"EEG feature dimension {EEG.shape[1]} is not divisible by EEG_CH={EEG_CH}"
        )

    if EMG.shape[1] % EMG_CH != 0:
        raise ValueError(
            f"EMG feature dimension {EMG.shape[1]} is not divisible by EMG_CH={EMG_CH}"
        )

    eeg_t = EEG.shape[1] // EEG_CH
    emg_t = EMG.shape[1] // EMG_CH

    print(f"[INFO] inferred EEG window: {EEG_CH} x {eeg_t}")
    print(f"[INFO] inferred EMG window: {EMG_CH} x {emg_t}")

    if eeg_t != EXPECTED_T:
        print(f"[WARNING] EEG inferred T={eeg_t}, expected {EXPECTED_T}")

    if emg_t != EXPECTED_T:
        print(f"[WARNING] EMG inferred T={emg_t}, expected {EXPECTED_T}")

    EEG = EEG.reshape(EEG.shape[0], EEG_CH, eeg_t)
    EMG = EMG.reshape(EMG.shape[0], EMG_CH, emg_t)

    return EEG, EMG, eeg_t, emg_t


# ============================================================
# TEMPORAL SYNCHRONIZATION — CROSS-CORRELATION ALIGNMENT
# Does not change model architecture.
# Applied before Dataset/DataLoader creation.
# Input shape:
#   EEG: (N, EEG_CH, T)
#   EMG: (N, EMG_CH, T)
# ============================================================

def _zscore_1d(x, eps=1e-8):
    x = np.asarray(x, dtype=np.float32)
    return (x - np.mean(x)) / (np.std(x) + eps)


def align_one_eeg_emg_window_crosscorr(eeg_win, emg_win, max_lag_samples=None):
    eeg_win = np.asarray(eeg_win, dtype=np.float32)
    emg_win = np.asarray(emg_win, dtype=np.float32)

    T = min(eeg_win.shape[1], emg_win.shape[1])
    eeg_win = eeg_win[:, :T].copy()
    emg_win = emg_win[:, :T].copy()

    eeg_ref = _zscore_1d(np.mean(eeg_win, axis=0))
    emg_ref = _zscore_1d(np.mean(emg_win, axis=0))

    if np.std(eeg_ref) < 1e-8 or np.std(emg_ref) < 1e-8:
        return eeg_win, emg_win, 0, 0.0

    corr = correlate(eeg_ref, emg_ref, mode="full", method="fft")
    lags = np.arange(-len(eeg_ref) + 1, len(emg_ref))

    denom = np.linalg.norm(eeg_ref) * np.linalg.norm(emg_ref)
    if denom > 1e-12:
        corr = corr / denom
    else:
        corr = np.zeros_like(corr)

    if max_lag_samples is not None:
        mask = np.abs(lags) <= int(max_lag_samples)
        corr_search = corr[mask]
        lags_search = lags[mask]
    else:
        corr_search = corr
        lags_search = lags

    if len(corr_search) == 0:
        return eeg_win, emg_win, 0, 0.0

    best_idx = int(np.argmax(corr_search))
    best_lag = int(lags_search[best_idx])
    sync_score = float(corr_search[best_idx])

    # Circular shifting for local alignment.
    # Window shape is channels × time, so shift along axis=1.
    if best_lag > 0:
        eeg_win = np.roll(eeg_win, best_lag, axis=1)
    elif best_lag < 0:
        emg_win = np.roll(emg_win, -best_lag, axis=1)

    return eeg_win, emg_win, best_lag, sync_score


def apply_crosscorr_sync_to_all_windows(EEG, EMG, sid=None, max_lag_samples=50):
    if EEG.ndim != 3 or EMG.ndim != 3:
        raise ValueError(f"Expected EEG/EMG as 3D arrays. Got EEG={EEG.shape}, EMG={EMG.shape}")

    N = min(EEG.shape[0], EMG.shape[0])
    T = min(EEG.shape[2], EMG.shape[2])

    EEG = EEG[:N, :, :T].astype(np.float32, copy=False)
    EMG = EMG[:N, :, :T].astype(np.float32, copy=False)

    EEG_sync = np.empty_like(EEG, dtype=np.float32)
    EMG_sync = np.empty_like(EMG, dtype=np.float32)

    lags = np.zeros(N, dtype=np.int32)
    scores = np.zeros(N, dtype=np.float32)

    for i in range(N):
        eeg_a, emg_a, lag, score = align_one_eeg_emg_window_crosscorr(
            EEG[i],
            EMG[i],
            max_lag_samples=max_lag_samples
        )

        EEG_sync[i] = eeg_a
        EMG_sync[i] = emg_a
        lags[i] = lag
        scores[i] = score

    report_global = {
        "scope": "global",
        "subject_id": -1,
        "n_windows": int(N),
        "lag_mean": float(np.mean(lags)),
        "lag_std": float(np.std(lags)),
        "lag_min": int(np.min(lags)),
        "lag_max": int(np.max(lags)),
        "sync_score_mean": float(np.mean(scores)),
        "sync_score_std": float(np.std(scores)),
        "sync_score_min": float(np.min(scores)),
        "sync_score_max": float(np.max(scores)),
        "max_lag_samples": -1 if max_lag_samples is None else int(max_lag_samples),
    }

    rows = [report_global]

    if sid is not None:
        sid_arr = np.asarray(sid)[:N]

        for s in np.unique(sid_arr):
            idx = sid_arr == s
            rows.append({
                "scope": "subject",
                "subject_id": int(s),
                "n_windows": int(np.sum(idx)),
                "lag_mean": float(np.mean(lags[idx])),
                "lag_std": float(np.std(lags[idx])),
                "lag_min": int(np.min(lags[idx])),
                "lag_max": int(np.max(lags[idx])),
                "sync_score_mean": float(np.mean(scores[idx])),
                "sync_score_std": float(np.std(scores[idx])),
                "sync_score_min": float(np.min(scores[idx])),
                "sync_score_max": float(np.max(scores[idx])),
                "max_lag_samples": -1 if max_lag_samples is None else int(max_lag_samples),
            })

    sync_report_df = pd.DataFrame(rows)
    sync_global = float(np.mean(scores))

    print("\n================ DATASET-1 CROSS-CORRELATION SYNC REPORT ================")
    print(f"Applied to windows: {N}")
    print(f"Lag mean/std: {report_global['lag_mean']:.4f} / {report_global['lag_std']:.4f}")
    print(f"Lag min/max: {report_global['lag_min']} / {report_global['lag_max']}")
    print(f"Sync score mean/std: {report_global['sync_score_mean']:.4f} / {report_global['sync_score_std']:.4f}")
    print("==========================================================================\n")

    return EEG_sync, EMG_sync, sync_report_df, sync_global


# ============================================================
# DATASET + AUGMENTATION
# ============================================================

def frequency_perturbation(x, min_factor=0.98, max_factor=1.02):
    x = np.asarray(x, dtype=np.float32)

    if x.ndim != 2:
        return x

    C, T = x.shape

    if T < 8:
        return x

    factor = np.random.uniform(min_factor, max_factor)
    new_T = max(8, int(T * factor))

    x_new = resample(x, new_T, axis=1)
    x_back = resample(x_new, T, axis=1)

    return x_back.astype(np.float32)


class EEGEMGWindowDataset(Dataset):
    def __init__(self, EEG, EMG, y, augment=False):
        self.EEG = EEG.astype(np.float32)
        self.EMG = EMG.astype(np.float32)
        self.y = y.astype(np.int64)
        self.augment = augment

    def __len__(self):
        return self.y.shape[0]

    def _augment_one(self, x):
        x = x.copy()

        # Gaussian noise
        if np.random.rand() < 0.50:
            x = x + np.random.normal(0.0, AUG_NOISE_STD, size=x.shape).astype(np.float32)

        # Amplitude scaling
        if np.random.rand() < 0.50:
            scale = np.random.uniform(AUG_SCALE_MIN, AUG_SCALE_MAX)
            x = x * scale

        # Time shifting
        if np.random.rand() < 0.50:
            shift = np.random.randint(-AUG_SHIFT_MAX, AUG_SHIFT_MAX + 1)
            x = np.roll(x, shift=shift, axis=1)

        # Frequency perturbation
        if USE_FREQ_PERTURBATION and np.random.rand() < FREQ_PERTURB_PROB:
            x = frequency_perturbation(
                x,
                min_factor=FREQ_PERTURB_MIN,
                max_factor=FREQ_PERTURB_MAX
            )

        return x.astype(np.float32)

    def __getitem__(self, idx):
        eeg = self.EEG[idx]
        emg = self.EMG[idx]
        y = self.y[idx]

        if self.augment:
            eeg = self._augment_one(eeg)
            emg = self._augment_one(emg)

        return (
            torch.tensor(eeg, dtype=torch.float32),
            torch.tensor(emg, dtype=torch.float32),
            torch.tensor(y, dtype=torch.long)
        )


# ============================================================
# PAPER-MATCHED NEUROFUSION-TRANS MODEL
# MODEL ARCHITECTURE UNCHANGED
# ============================================================

class LearnablePositionalEncoding(nn.Module):
    def __init__(self, max_len, d_model):
        super().__init__()
        self.pos = nn.Parameter(torch.zeros(1, max_len, d_model))
        nn.init.trunc_normal_(self.pos, std=0.02)

    def forward(self, x):
        return x + self.pos[:, :x.size(1), :]


class NeuroFusionTransPaper(nn.Module):
    def __init__(
        self,
        eeg_ch,
        emg_ch,
        seq_len,
        d_model,
        num_heads,
        num_layers,
        ff_dim,
        num_classes,
        dropout=0.2
    ):
        super().__init__()

        self.eeg_input_proj = nn.Linear(eeg_ch, d_model)
        self.emg_input_proj = nn.Linear(emg_ch, d_model)

        self.eeg_pos = LearnablePositionalEncoding(seq_len, d_model)
        self.emg_pos = LearnablePositionalEncoding(seq_len, d_model)

        enc_layer_eeg = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=num_heads,
            dim_feedforward=ff_dim,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True
        )

        enc_layer_emg = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=num_heads,
            dim_feedforward=ff_dim,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True
        )

        self.eeg_encoder = nn.TransformerEncoder(enc_layer_eeg, num_layers=num_layers)
        self.emg_encoder = nn.TransformerEncoder(enc_layer_emg, num_layers=num_layers)

        init_alpha = torch.tensor([INIT_ALPHA_EEG, INIT_ALPHA_EMG], dtype=torch.float32)
        self.cross_attention_logits = nn.Parameter(torch.log(init_alpha))

        self.cross_attention = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )

        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

        self.fc = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, num_classes)
        )

    def forward(self, eeg, emg):
        # eeg/emg: (B, C, T)
        eeg = eeg.transpose(1, 2)  # (B, T, C)
        emg = emg.transpose(1, 2)

        eeg = self.eeg_input_proj(eeg)
        emg = self.emg_input_proj(emg)

        eeg = self.eeg_pos(eeg)
        emg = self.emg_pos(emg)

        h_eeg = self.eeg_encoder(eeg)
        h_emg = self.emg_encoder(emg)

        alpha = torch.softmax(self.cross_attention_logits, dim=0)
        h_fusion = alpha[0] * h_eeg + alpha[1] * h_emg

        h_cross, attn_weights = self.cross_attention(
            h_fusion,
            h_fusion,
            h_fusion,
            need_weights=True
        )

        h_cross = self.norm(h_cross + h_fusion)

        pooled = h_cross.mean(dim=1)
        pooled = self.dropout(pooled)

        logits = self.fc(pooled)

        return logits, alpha, attn_weights


# ============================================================
# MIXUP — SYNTHETIC SAMPLE GENERATION
# ============================================================

def soft_cross_entropy(logits, soft_targets):
    log_probs = F.log_softmax(logits, dim=1)
    return -(soft_targets * log_probs).sum(dim=1).mean()


def maybe_mixup(eeg, emg, y, num_classes):
    if not USE_MIXUP:
        return eeg, emg, F.one_hot(y, num_classes=num_classes).float(), False

    if np.random.rand() > 0.50:
        return eeg, emg, F.one_hot(y, num_classes=num_classes).float(), False

    lam = np.random.beta(MIXUP_ALPHA, MIXUP_ALPHA)
    idx = torch.randperm(eeg.size(0), device=eeg.device)

    eeg_mix = lam * eeg + (1.0 - lam) * eeg[idx]
    emg_mix = lam * emg + (1.0 - lam) * emg[idx]

    y_one = F.one_hot(y, num_classes=num_classes).float()
    y_mix = lam * y_one + (1.0 - lam) * y_one[idx]

    return eeg_mix, emg_mix, y_mix, True


# ============================================================
# METRICS
# ============================================================

def evaluate_no_plots(model, loader, num_classes, device, return_cm=False):
    model.eval()

    y_true, y_pred, y_scores = [], [], []
    alpha_list = []

    with torch.no_grad():
        for eeg, emg, yb in loader:
            eeg = eeg.to(device, non_blocking=True)
            emg = emg.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)

            logits, alpha, _ = model(eeg, emg)

            probs = torch.softmax(logits, dim=1)
            pred = torch.argmax(probs, dim=1)

            y_true.extend(yb.cpu().numpy().tolist())
            y_pred.extend(pred.cpu().numpy().tolist())
            y_scores.extend(probs.cpu().numpy().tolist())
            alpha_list.append(alpha.detach().cpu().numpy())

    y_true = np.asarray(y_true, dtype=np.int64)
    y_pred = np.asarray(y_pred, dtype=np.int64)
    y_scores = np.asarray(y_scores, dtype=np.float64)

    labels_all = np.arange(num_classes, dtype=np.int64)

    precision = precision_score(y_true, y_pred, average="weighted", zero_division=0)
    recall = recall_score(y_true, y_pred, average="weighted", zero_division=0)
    f1 = f1_score(y_true, y_pred, average="weighted", zero_division=0)
    acc = accuracy_score(y_true, y_pred)
    bal_acc = balanced_accuracy_score(y_true, y_pred)
    kappa = cohen_kappa_score(y_true, y_pred)
    mcc = matthews_corrcoef(y_true, y_pred)

    try:
        ll = log_loss(y_true, y_scores, labels=labels_all)
    except Exception:
        ll = np.nan

    try:
        top3 = top_k_accuracy_score(
            y_true,
            y_scores,
            k=min(3, num_classes),
            labels=labels_all
        )
    except Exception:
        top3 = np.nan

    cm = confusion_matrix(y_true, y_pred, labels=labels_all)

    with np.errstate(divide="ignore", invalid="ignore"):
        per_class_acc = np.diag(cm) / np.maximum(1, cm.sum(axis=1))

    mpce = float(np.mean(1.0 - per_class_acc))

    auroc = None
    auprc = None

    try:
        Yb = label_binarize(y_true, classes=labels_all)
        auroc = roc_auc_score(Yb, y_scores, average="weighted")
        auprc = average_precision_score(Yb, y_scores, average="weighted")
    except Exception:
        pass

    if len(alpha_list) > 0:
        alpha_mean = np.mean(np.vstack(alpha_list), axis=0)
        alpha_eeg = float(alpha_mean[0])
        alpha_emg = float(alpha_mean[1])
    else:
        alpha_eeg, alpha_emg = np.nan, np.nan

    print(f"Prec={precision:.3f} Rec={recall:.3f} F1={f1:.3f} Acc={acc:.3f} BalAcc={bal_acc:.3f}")
    print(f"Kappa={kappa:.3f} MCC={mcc:.3f} LogLoss={ll:.4f} Top3={top3:.4f} MPCE={mpce:.4f}")
    print(f"Alpha EEG={alpha_eeg:.3f} | Alpha EMG={alpha_emg:.3f}")

    if auroc is not None:
        print(f"AUROC={auroc:.4f} AUPRC={auprc:.4f}")

    out = {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "balanced_accuracy": bal_acc,
        "kappa": kappa,
        "mcc": mcc,
        "log_loss": ll,
        "top3": top3,
        "mpce": mpce,
        "auroc": auroc,
        "auprc": auprc,
        "alpha_eeg": alpha_eeg,
        "alpha_emg": alpha_emg,
    }

    if return_cm:
        out["confusion_matrix"] = cm

    return out


# ============================================================
# TRAIN
# ============================================================

def train_model(
    model,
    train_loader,
    val_loader,
    device,
    num_classes,
    epochs=150,
    patience=25,
    lr=1e-3,
    weight_decay=1e-4,
    use_amp=True
):
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )

    scheduler = torch.optim.lr_scheduler.CyclicLR(
        optimizer,
        base_lr=lr / 10,
        max_lr=lr,
        step_size_up=max(1, len(train_loader) * 2),
        mode="triangular2",
        cycle_momentum=False
    )

    scaler = torch.cuda.amp.GradScaler(enabled=(use_amp and device.type == "cuda"))
    ce = nn.CrossEntropyLoss()

    best_loss = float("inf")
    best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    bad = 0

    for ep in range(1, epochs + 1):
        model.train()
        tr_loss = 0.0

        for eeg, emg, yb in train_loader:
            eeg = eeg.to(device, non_blocking=True)
            emg = emg.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            eeg_in, emg_in, soft_y, mixed = maybe_mixup(eeg, emg, yb, num_classes)

            with torch.cuda.amp.autocast(enabled=(use_amp and device.type == "cuda")):
                logits, _, _ = model(eeg_in, emg_in)
                loss = soft_cross_entropy(logits, soft_y) if mixed else ce(logits, yb)

            scaler.scale(loss).backward()

            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)

            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            tr_loss += float(loss.detach().cpu())

        model.eval()
        va_loss = 0.0

        with torch.no_grad():
            for eeg, emg, yb in val_loader:
                eeg = eeg.to(device, non_blocking=True)
                emg = emg.to(device, non_blocking=True)
                yb = yb.to(device, non_blocking=True)

                with torch.cuda.amp.autocast(enabled=(use_amp and device.type == "cuda")):
                    logits, _, _ = model(eeg, emg)
                    loss = ce(logits, yb)

                va_loss += float(loss.detach().cpu())

        tr_loss /= max(1, len(train_loader))
        va_loss /= max(1, len(val_loader))

        print(f"Epoch {ep}/{epochs} | train={tr_loss:.4f} | val={va_loss:.4f}")

        if va_loss < best_loss:
            best_loss = va_loss
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                print("Early stopping.")
                break

    model.load_state_dict(best_state)
    return model


# ============================================================
# ONLINE ADAPTATION
# ============================================================

def online_adapt(
    model,
    buffer_EEG,
    buffer_EMG,
    buffer_y,
    replay_EEG,
    replay_EMG,
    replay_y,
    val_loader,
    device,
    num_cycles=5,
    batch_size=32,
    lr=1e-5,
    use_amp=True
):
    if replay_EEG is not None and len(replay_EEG) > 0:
        buffer_EEG = np.concatenate([buffer_EEG, replay_EEG], axis=0)
        buffer_EMG = np.concatenate([buffer_EMG, replay_EMG], axis=0)
        buffer_y = np.concatenate([buffer_y, replay_y], axis=0)

    ds = EEGEMGWindowDataset(
        buffer_EEG,
        buffer_EMG,
        buffer_y,
        augment=True
    )

    loader = DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY
    )

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr,
        weight_decay=WEIGHT_DECAY
    )

    scheduler = torch.optim.lr_scheduler.CyclicLR(
        optimizer,
        base_lr=lr / 10,
        max_lr=lr,
        step_size_up=5,
        mode="triangular2",
        cycle_momentum=False
    )

    scaler = torch.cuda.amp.GradScaler(enabled=(use_amp and device.type == "cuda"))
    ce = nn.CrossEntropyLoss()

    theta_prev = {
        name: p.detach().clone()
        for name, p in model.named_parameters()
        if p.requires_grad
    }

    best_val = float("inf")
    best_state = {
        k: v.detach().cpu().clone()
        for k, v in model.state_dict().items()
    }

    for c in range(1, num_cycles + 1):
        print(f"Online cycle {c}/{num_cycles}")

        model.train()

        for eeg, emg, yb in loader:
            eeg = eeg.to(device, non_blocking=True)
            emg = emg.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(enabled=(use_amp and device.type == "cuda")):
                logits, _, _ = model(eeg, emg)
                loss = ce(logits, yb)

                reg = 0.0
                for name, p in model.named_parameters():
                    if p.requires_grad:
                        reg = reg + torch.sum((p - theta_prev[name]) ** 2)

                loss = loss + ADAPT_L2_LAMBDA * reg

            scaler.scale(loss).backward()

            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)

            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

        model.eval()
        val_loss = 0.0

        with torch.no_grad():
            for eeg, emg, yb in val_loader:
                eeg = eeg.to(device, non_blocking=True)
                emg = emg.to(device, non_blocking=True)
                yb = yb.to(device, non_blocking=True)

                with torch.cuda.amp.autocast(enabled=(use_amp and device.type == "cuda")):
                    logits, _, _ = model(eeg, emg)
                    loss = ce(logits, yb)

                val_loss += float(loss.detach().cpu())

        val_loss /= max(1, len(val_loader))
        print(f"  val_loss={val_loss:.4f}")

        if val_loss < best_val:
            best_val = val_loss
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }
        else:
            print("  early stop online adapt")
            break

    model.load_state_dict(best_state)
    return model


# ============================================================
# MAIN
# ============================================================

if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    if device.type == "cuda":
        print(f"[INFO] CUDA: {torch.cuda.get_device_name(0)}")
        torch.backends.cudnn.benchmark = True
        try:
            torch.set_float32_matmul_precision("high")
        except Exception:
            pass
    else:
        print("[INFO] Running on CPU")

    EEG_flat, EMG_flat, y, sid = load_all_subjects(DATA_ROOT)
    EEG, EMG, eeg_t, emg_t = reshape_flat_windows(EEG_flat, EMG_flat)

    if eeg_t != emg_t:
        raise ValueError(f"EEG and EMG sequence length mismatch: EEG={eeg_t}, EMG={emg_t}")

    sync_global = np.nan

    if APPLY_CROSSCORR_SYNC:
        EEG, EMG, sync_report_df, sync_global = apply_crosscorr_sync_to_all_windows(
            EEG,
            EMG,
            sid=sid,
            max_lag_samples=MAX_SYNC_LAG_SAMPLES
        )

        if SAVE_SYNC_REPORT:
            sync_report_path = OUT_DIR / "crosscorr_sync_report_dataset1.csv"
            sync_report_df.to_csv(sync_report_path, index=False)
            print(f"[SAVED] Cross-correlation sync report: {sync_report_path}")

    num_classes = int(np.max(y)) + 1

    print(f"[INFO] EEG={EEG.shape} EMG={EMG.shape} y={y.shape}")
    print(f"[INFO] classes={num_classes}")
    print(f"[INFO] label counts={np.bincount(y, minlength=num_classes)}")
    print(f"[INFO] subjects loaded={np.unique(sid)}")
    print(f"[INFO] value range EEG=({EEG.min():.4f}, {EEG.max():.4f}) EMG=({EMG.min():.4f}, {EMG.max():.4f})")
    print(f"[INFO] sync_global={sync_global}")

    if len(np.unique(sid)) <= 1:
        print("[WARNING] Only one subject loaded. For all-subject Dataset-1 results, run Dataset-1 preprocessing for all subjects first.")

    skf = StratifiedKFold(
        n_splits=K_FOLDS,
        shuffle=True,
        random_state=SEED
    )

    all_metrics = []
    best_acc = -1.0
    best_fold = None
    best_state = None

    for fold, (tr_idx, va_idx) in enumerate(skf.split(EEG, y), start=1):
        print(f"\n================ Fold {fold}/{K_FOLDS} ================")

        EEG_tr = EEG[tr_idx]
        EMG_tr = EMG[tr_idx]
        y_tr = y[tr_idx]

        EEG_va = EEG[va_idx]
        EMG_va = EMG[va_idx]
        y_va = y[va_idx]

        train_ds = EEGEMGWindowDataset(
            EEG_tr,
            EMG_tr,
            y_tr,
            augment=USE_TRAIN_AUGMENTATION
        )

        val_ds = EEGEMGWindowDataset(
            EEG_va,
            EMG_va,
            y_va,
            augment=False
        )

        train_loader = DataLoader(
            train_ds,
            batch_size=BATCH_SIZE,
            shuffle=True,
            num_workers=NUM_WORKERS,
            pin_memory=PIN_MEMORY
        )

        val_loader = DataLoader(
            val_ds,
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=NUM_WORKERS,
            pin_memory=PIN_MEMORY
        )

        model = NeuroFusionTransPaper(
            eeg_ch=EEG_CH,
            emg_ch=EMG_CH,
            seq_len=eeg_t,
            d_model=D_MODEL,
            num_heads=NUM_HEADS,
            num_layers=NUM_LAYERS,
            ff_dim=FF_DIM,
            num_classes=num_classes,
            dropout=DROPOUT
        ).to(device)

        model = train_model(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            device=device,
            num_classes=num_classes,
            epochs=EPOCHS,
            patience=PATIENCE,
            lr=LR,
            weight_decay=WEIGHT_DECAY,
            use_amp=USE_AMP
        )

        print("Before online adapt:")
        before_met = evaluate_no_plots(
            model=model,
            loader=val_loader,
            num_classes=num_classes,
            device=device,
            return_cm=False
        )

        if DO_ONLINE_ADAPT:
            online_n = min(ONLINE_NEW_N, len(EEG_va))
            replay_n = min(REPLAY_N, len(EEG_tr))

            print(
                f"[ONLINE ADAPT] online_n={online_n} | "
                f"replay_n={replay_n} | batch_size={ONLINE_BS}"
            )

            if online_n > 0:
                model = online_adapt(
                    model=model,
                    buffer_EEG=EEG_va[:online_n],
                    buffer_EMG=EMG_va[:online_n],
                    buffer_y=y_va[:online_n],
                    replay_EEG=EEG_tr[:replay_n],
                    replay_EMG=EMG_tr[:replay_n],
                    replay_y=y_tr[:replay_n],
                    val_loader=val_loader,
                    device=device,
                    num_cycles=ONLINE_CYCLES,
                    batch_size=ONLINE_BS,
                    lr=ONLINE_LR,
                    use_amp=USE_AMP
                )

        print("After online adapt:")
        after_met = evaluate_no_plots(
            model=model,
            loader=val_loader,
            num_classes=num_classes,
            device=device,
            return_cm=True
        )

        cm = after_met.pop("confusion_matrix")

        cm_path = OUT_DIR / f"confusion_matrix_fold{fold}.csv"
        pd.DataFrame(cm).to_csv(cm_path, index=False)

        met_row = {
            "fold": fold,

            "before_accuracy": before_met["accuracy"],
            "before_precision": before_met["precision"],
            "before_recall": before_met["recall"],
            "before_f1": before_met["f1"],
            "before_balanced_accuracy": before_met["balanced_accuracy"],
            "before_kappa": before_met["kappa"],
            "before_mcc": before_met["mcc"],

            "accuracy": after_met["accuracy"],
            "precision": after_met["precision"],
            "recall": after_met["recall"],
            "f1": after_met["f1"],
            "balanced_accuracy": after_met["balanced_accuracy"],
            "kappa": after_met["kappa"],
            "mcc": after_met["mcc"],
            "log_loss": after_met["log_loss"],
            "top3": after_met["top3"],
            "mpce": after_met["mpce"],
            "auroc": after_met["auroc"],
            "auprc": after_met["auprc"],
            "alpha_eeg": after_met["alpha_eeg"],
            "alpha_emg": after_met["alpha_emg"],

            "sync_score_global": sync_global,
            "crosscorr_sync_applied": bool(APPLY_CROSSCORR_SYNC),
            "online_new_n": int(online_n if DO_ONLINE_ADAPT else 0),
            "replay_n": int(replay_n if DO_ONLINE_ADAPT else 0),
            "frequency_perturbation_used": bool(USE_FREQ_PERTURBATION),

            "n_train": len(tr_idx),
            "n_val": len(va_idx),
            "n_subjects_loaded": int(len(np.unique(sid))),
        }

        all_metrics.append(met_row)

        fold_model_path = OUT_DIR / f"NeuroFusionTrans_dataset1_allsubjects_fold{fold}.pth"

        torch.save(
            {
                "state_dict": model.state_dict(),
                "num_classes": num_classes,
                "eeg_ch": EEG_CH,
                "emg_ch": EMG_CH,
                "seq_len": eeg_t,
                "d_model": D_MODEL,
                "num_heads": NUM_HEADS,
                "num_layers": NUM_LAYERS,
                "ff_dim": FF_DIM,
                "dropout": DROPOUT,
                "label_set": np.unique(y).tolist(),
                "subjects_loaded": np.unique(sid).tolist(),
                "seed": SEED,
                "fold": fold,
                "metrics": met_row,
                "crosscorr_sync_applied": bool(APPLY_CROSSCORR_SYNC),
                "sync_score_global": sync_global,
                "online_new_n": int(online_n if DO_ONLINE_ADAPT else 0),
                "replay_n": int(replay_n if DO_ONLINE_ADAPT else 0),
                "frequency_perturbation_used": bool(USE_FREQ_PERTURBATION),
            },
            fold_model_path
        )

        print(f"[SAVED] {fold_model_path}")
        print(f"[SAVED] {cm_path}")

        if after_met["accuracy"] > best_acc:
            best_acc = after_met["accuracy"]
            best_fold = fold
            best_state = {
                "state_dict": model.state_dict(),
                "num_classes": num_classes,
                "eeg_ch": EEG_CH,
                "emg_ch": EMG_CH,
                "seq_len": eeg_t,
                "d_model": D_MODEL,
                "num_heads": NUM_HEADS,
                "num_layers": NUM_LAYERS,
                "ff_dim": FF_DIM,
                "dropout": DROPOUT,
                "best_fold": best_fold,
                "label_set": np.unique(y).tolist(),
                "subjects_loaded": np.unique(sid).tolist(),
                "seed": SEED,
                "metrics": met_row,
                "crosscorr_sync_applied": bool(APPLY_CROSSCORR_SYNC),
                "sync_score_global": sync_global,
                "online_new_n": int(online_n if DO_ONLINE_ADAPT else 0),
                "replay_n": int(replay_n if DO_ONLINE_ADAPT else 0),
                "frequency_perturbation_used": bool(USE_FREQ_PERTURBATION),
            }

    metrics_df = pd.DataFrame(all_metrics)

    metrics_path = OUT_DIR / "fold_metrics_NeuroFusionTrans_dataset1_allsubjects.csv"
    metrics_df.to_csv(metrics_path, index=False)

    best_path = OUT_DIR / "NeuroFusionTrans_dataset1_allsubjects_best.pth"
    torch.save(best_state, best_path)

    print("\n================ FINAL SUMMARY ================")
    print(metrics_df)

    print("\n[MEAN ± STD AFTER ONLINE ADAPT]")
    for col in [
        "accuracy",
        "precision",
        "recall",
        "f1",
        "balanced_accuracy",
        "kappa",
        "mcc",
        "log_loss",
        "top3",
        "mpce",
        "auroc",
        "auprc",
        "alpha_eeg",
        "alpha_emg",
        "sync_score_global",
    ]:
        if col in metrics_df.columns:
            vals = pd.to_numeric(metrics_df[col], errors="coerce")
            print(f"{col}: {vals.mean():.4f} ± {vals.std():.4f}")

    print("\n[MEAN ± STD BEFORE ONLINE ADAPT]")
    for col in [
        "before_accuracy",
        "before_precision",
        "before_recall",
        "before_f1",
        "before_balanced_accuracy",
        "before_kappa",
        "before_mcc",
    ]:
        if col in metrics_df.columns:
            vals = pd.to_numeric(metrics_df[col], errors="coerce")
            print(f"{col}: {vals.mean():.4f} ± {vals.std():.4f}")

    print(f"\n[OK] Saved fold metrics: {metrics_path}")
    print(f"[OK] Saved best model: {best_path}")
    print(f"[OK] Best fold: {best_fold} | acc={best_acc:.4f}")